# CPS Smoke Test + MT Non-Regression Check

Validates CPS implementation before full run matrix.
1. CPS smoke test: cps_r10 seed_0 with semi_start_epoch=1
2. MT non-regression: mean_teacher_r10 seed_0 in fresh exp_dir


In [ ]:
# ============================================================
# SETUP — run this cell once after every runtime restart
# Do NOT run training here. Select one experiment cell below.
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

!git clone https://github.com/sebastianquispearias/tesis-seg.git
%cd tesis-seg
!pip install -q -r requirements.txt

# --- Environment fingerprint (for reproducibility debugging) ---
import torch, sys
print("Python    :", sys.version)
print("torch     :", torch.__version__)
print("CUDA      :", torch.version.cuda)
!git log --oneline -1
!pip show albumentations | grep Version
!nvidia-smi | grep -E "NVIDIA|Driver Version|CUDA Version"
# --------------------------------------------------------------

import sys
sys.path.append("/content/tesis-seg")

from src.defaults import get_default_config, summarize_config
from src.augmentations import (
    get_supervised_train_augmentation,
    get_weak_augmentation,
    get_strong_augmentation,
)
from src.datasets import (
    build_supervised_datasets,
    build_unlabeled_datasets,
    build_dataloaders,
)
from src.train import run_training
from src.evaluate import evaluate_checkpoint
from src.visualization import show_dataset_examples, show_predictions

# ── Debug fingerprint helpers ─────────────────────────────────
import json, os, platform, subprocess, time, importlib.metadata

def _fp_get_version(pkg):
    try: return importlib.metadata.version(pkg)
    except Exception: return None

def _fp_git_hash_from_src(src_train_file):
    # Derive repo root from src/train.py: {repo_root}/src/train.py
    repo_root = os.path.dirname(os.path.dirname(os.path.abspath(src_train_file)))
    try:
        return subprocess.check_output(
            ["git", "log", "--oneline", "-1"], cwd=repo_root, stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception: return None

def _fp_save(pre, post, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump({"pre_run": pre, "post_run": post}, f, indent=2, default=str)

def _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                    unlabeled_ds=None, temporal_unlab_ds=None):
    import torch, sys
    from src.models import create_model

    # 1. Environment
    try: gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "n/a"
    except: gpu_name = "n/a"
    env = {
        "python":      sys.version,
        "torch":       torch.__version__,
        "torchvision": _fp_get_version("torchvision"),
        "cuda":        torch.version.cuda,
        "cudnn":       str(torch.backends.cudnn.version()) if torch.cuda.is_available() else "n/a",
        "segmentation_models_pytorch": _fp_get_version("segmentation-models-pytorch"),
        "albumentations": _fp_get_version("albumentations"),
        "platform":    platform.platform(),
        "gpu_name":    gpu_name,
    }

    # 2. Code provenance — git hash derived from actual runtime src path
    import src.train, src.datasets, src.defaults, src.evaluate
    provenance = {
        "src_train":    src.train.__file__,
        "src_datasets": src.datasets.__file__,
        "src_defaults": src.defaults.__file__,
        "src_evaluate": src.evaluate.__file__,
        "git_hash":     _fp_git_hash_from_src(src.train.__file__),
    }

    # 3. Effective config
    cfg_keys = [
        "seed", "arch", "backbone", "n_classes",
        "image_preproc", "mask_smoothing", "target_size", "use_pad", "imagenet_norm",
        "batch_size", "num_workers", "drop_last", "num_augmented",
        "lr", "weight_decay", "epochs", "warmup_epochs", "patience_es", "eval_threshold",
        "use_semi", "use_temp_consistency",
        "lambda_u", "tau", "ema_decay", "semi_start_epoch", "semi_warmup_epochs", "lambda_t",
        "unlabeled_subdir", "exp_dir",
    ]
    eff_cfg = {k: cfg.get(k) for k in cfg_keys}
    unlab_loader = loaders.get("unlabeled_loader")
    eff_cfg["batch_size_unlab"] = unlab_loader.batch_size if unlab_loader is not None else None

    # 4. Dataset / loader facts
    ds_facts = {
        "len_train_ds":          len(train_ds),
        "len_val_ds":            len(val_ds),
        "len_test_ds":           len(test_ds),
        "len_unlabeled_ds":      len(unlabeled_ds) if unlabeled_ds is not None else None,
        "len_temporal_unlab_ds": len(temporal_unlab_ds) if temporal_unlab_ds is not None else None,
        "train_loader_batch_size":      loaders["train_loader"].batch_size,
        "train_loader_num_workers":     loaders["train_loader"].num_workers,
        "train_loader_drop_last":       loaders["train_loader"].drop_last,
        "unlabeled_loader_batch_size":  unlab_loader.batch_size if unlab_loader else None,
        "unlabeled_loader_num_workers": unlab_loader.num_workers if unlab_loader else None,
        "unlabeled_loader_drop_last":   unlab_loader.drop_last if unlab_loader else None,
    }

    # 5. Sample identifiers — reads .files attribute, no IO beyond what dataset already did
    try: sup_ids = train_ds.files[:5]
    except Exception as e: sup_ids = f"unavailable: {e}"
    try: unl_ids = unlabeled_ds.files[:5] if unlabeled_ds is not None else None
    except Exception as e: unl_ids = f"unavailable: {e}"
    sample_ids = {"first5_train": sup_ids, "first5_unlabeled": unl_ids}

    # 6. Batch tensor shapes — analytical, no DataLoader consumed, no RNG touched
    try:
        H, W = cfg["target_size"]
        C = 3  # IMREAD_COLOR: grayscale PNGs expand to 3 identical channels
        eff_bs = cfg["batch_size"] * (1 + cfg.get("num_augmented", 0))  # flatten_collate
        bs_u = max(1, cfg["batch_size"] // 4)  # mirrors datasets.py build_dataloaders
        batch_shapes = {
            "xb":   [eff_bs, C, H, W],
            "yb":   [eff_bs, 1, H, W],
            "xw_u": [bs_u, C, H, W] if unlab_loader is not None else None,
            "xs_u": [bs_u, C, H, W] if unlab_loader is not None else None,
            "note": "analytically derived from cfg — no DataLoader consumed",
        }
    except Exception as e:
        batch_shapes = {"error": str(e)}

    # 7. Model fingerprint — RNG save/restore so training is unaffected.
    # Belt-and-suspenders: run_training() also calls seed_everything(seed) first.
    try:
        import random as _random, numpy as _np
        _rng = {
            "py":   _random.getstate(),
            "np":   _np.random.get_state(),
            "th":   torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        }
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        model_fp = {
            "total_params":          sum(p.numel() for p in _m.parameters()),
            "trainable_params":      sum(p.numel() for p in _m.parameters() if p.requires_grad),
            "first_state_dict_keys": list(_m.state_dict().keys())[:8],
        }
        del _m
        _random.setstate(_rng["py"])
        _np.random.set_state(_rng["np"])
        torch.set_rng_state(_rng["th"])
        if _rng["cuda"] is not None:
            torch.cuda.set_rng_state_all(_rng["cuda"])
    except Exception as e:
        model_fp = {"error": str(e)}

    return {
        "timestamp_utc":     time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "environment":       env,
        "provenance":        provenance,
        "effective_cfg":     eff_cfg,
        "dataset_facts":     ds_facts,
        "sample_ids":        sample_ids,
        "batch_shapes":      batch_shapes,
        "model_fingerprint": model_fp,
    }


def _fp_collect_post(artifacts, results):
    history = artifacts.get("history") or []
    best_row = max(history, key=lambda r: r.get("val_iou_global", 0.0)) if history else None
    vm = (results or {}).get("val_metrics", {})
    tm = (results or {}).get("test_metrics", {})
    return {
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "best_path":     artifacts.get("best_path"),
        "best_epoch_info": {
            "epoch":          best_row.get("epoch") if best_row else None,
            "val_iou_global": best_row.get("val_iou_global") if best_row else None,
            "val_loss":       best_row.get("val_loss") if best_row else None,
        },
        "val_metrics": {
            "f1_global":       vm.get("global_f1"),
            "iou_global":      vm.get("global_iou"),
            "f1_sample_mean":  vm.get("sample_mean_f1"),
            "iou_sample_mean": vm.get("sample_mean_iou"),
        },
        "test_metrics": {
            "f1_global":       tm.get("global_f1"),
            "iou_global":      tm.get("global_iou"),
            "f1_sample_mean":  tm.get("sample_mean_f1"),
            "iou_sample_mean": tm.get("sample_mean_iou"),
        },
        "finished_successfully": True,
        "exception": None,
    }
# ──────────────────────────────────────────────────────────────

print("Imports OK — select one experiment cell below and run it.")

In [ ]:
# === ENVIRONMENT SANITY CHECK (UNM) ===
# Run this ONCE after the setup cell. Verifies Drive access, data dirs,
# GPU, key packages, and creates the output root. Raises on any failure
# so training never starts against a half-wired environment.
import os, sys

_errors = []
_warnings = []

# 1. Google Drive mount
_drive_root = "/content/drive/MyDrive"
if not os.path.isdir(_drive_root):
    _errors.append(f"Google Drive not mounted at {_drive_root}")

# 2. Data directories (img_root + unlabeled pool parents)
_img_root = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
if not os.path.isdir(_img_root):
    _errors.append(f"img_root missing: {_img_root}")

_pool_parents = [
    "unlabeling_r3_max0",
    "unlabeling_r5_max0",
    "unlabeling_r7_max0",
    "unlabeling_r10_max0",
    "unlabeling_r15_max0",
    "unlabeling_r20_max0",
    "unlabeling_std_matched_r3",
    "unlabeling_std_matched_r5",
    "unlabeling_std_matched_r7",
    "unlabeling_std_matched_r10",
    "unlabeling_std_matched_r15",
    "unlabeling_std_matched_r20",
    "unlabeling_all_lateral",
]
_missing_pools = []
for _p in _pool_parents:
    _full = os.path.join(_img_root, _p)
    if not os.path.isdir(_full):
        _missing_pools.append(_p)
if _missing_pools:
    _errors.append(f"{len(_missing_pools)} unlabeled pool dirs missing under {_img_root}: {_missing_pools}")

# 3. GPU
try:
    import torch
    if not torch.cuda.is_available():
        _errors.append("CUDA not available — training will fail or be unusable")
    else:
        print(f"[GPU] {torch.cuda.get_device_name(0)} | CUDA {torch.version.cuda} | torch {torch.__version__}")
except Exception as _e:
    _errors.append(f"torch import failed: {_e}")

# 4. Key packages (import + print version)
_pkgs = [("segmentation_models_pytorch", "smp"), ("albumentations", "A")]
for _pkg, _alias in _pkgs:
    try:
        _m = __import__(_pkg)
        _v = getattr(_m, "__version__", "unknown")
        print(f"[pkg] {_pkg}: {_v}")
    except Exception as _e:
        _errors.append(f"cannot import {_pkg}: {_e}")

# 5. Create output root (non-destructive; exist_ok=True)
_output_root = "/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1"
try:
    os.makedirs(_output_root, exist_ok=True)
    print(f"[output] root ready: {_output_root}")
except Exception as _e:
    _errors.append(f"cannot create output root {_output_root}: {_e}")

if _errors:
    print("\n*** ENVIRONMENT SANITY CHECK FAILED ***")
    for _err in _errors:
        print(f"  - {_err}")
    raise RuntimeError(f"{len(_errors)} environment check(s) failed — fix them before running any experiment cell.")

print("\n[OK] environment sanity check passed. You may run the experiment cells below.")


In [ ]:
# === CPS SMOKE TEST: cps_r10/seed_0 (semi_start_epoch=1 for quick activation) ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
# Paths (UNM)
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "cps_r10_smoketest"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

# Architecture (same as all other experiments)
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# CPS-specific settings
cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "cps"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 1    # SMOKE TEST: activate CPS immediately
cfg["semi_warmup_epochs"]   = 5    # SMOKE TEST: short warmup
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training (short run for smoke test)
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 30       # SMOKE TEST: just 30 epochs
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

# Unlabeled pool
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

# Disable expensive outputs for smoke test
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = False

print(summarize_config(cfg))

train_tf = get_supervised_train_augmentation(cfg)
weak_tf   = get_weak_augmentation(cfg)
strong_tf = get_strong_augmentation(cfg)

train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(cfg, weak_tf=weak_tf, strong_tf=strong_tf)
loaders = build_dataloaders(cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
                            unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds)

print(f"\nStarting CPS smoke test...")
artifacts = run_training(cfg, loaders)
print(f"\nSmoke test training complete. Best path: {artifacts['best_path']}")


In [ ]:
# === VERIFY CPS DIAGNOSTICS ===
import pandas as pd
import os

exp_dir = cfg["exp_dir"]
diag_path = os.path.join(exp_dir, "diagnostics_epoch.csv")

print(f"Reading: {diag_path}")
df = pd.read_csv(diag_path)

print(f"\nTotal epochs: {len(df)}")
print(f"Columns: {list(df.columns)}")

# Check CPS columns exist and are populated
cps_cols = ["cps_loss_A", "cps_loss_B", "sup_loss_B", "pseudo_agree",
            "pseudo_A_pos", "pseudo_B_pos", "val_iou_B"]
print(f"\n=== CPS Column Verification ===")
for col in cps_cols:
    if col in df.columns:
        vals = df[col]
        nonzero = (vals != 0).sum()
        print(f"  {col:15s}: present, nonzero in {nonzero}/{len(df)} epochs, last={vals.iloc[-1]:.6f}")
    else:
        print(f"  {col:15s}: MISSING!")

# Check CPS activated after semi_start_epoch
semi_start = cfg.get("semi_start_epoch", 1)
active_epochs = df[df["epoch"] >= semi_start]
if len(active_epochs) > 0:
    print(f"\n=== CPS Active Epochs (>= {semi_start}) ===")
    print(f"  cps_loss_A mean: {active_epochs['cps_loss_A'].mean():.6f}")
    print(f"  cps_loss_B mean: {active_epochs['cps_loss_B'].mean():.6f}")
    print(f"  sup_loss_B mean: {active_epochs['sup_loss_B'].mean():.6f}")
    print(f"  pseudo_agree mean: {active_epochs['pseudo_agree'].mean():.4f}")
    print(f"  pseudo_A_pos mean: {active_epochs['pseudo_A_pos'].mean():.4f}")
    print(f"  pseudo_B_pos mean: {active_epochs['pseudo_B_pos'].mean():.4f}")
    print(f"  val_iou_B mean: {active_epochs['val_iou_B'].mean():.4f}")

# PASS/FAIL Checks
print(f"\n=== PASS/FAIL Checks ===")
checks = []

# Check 1: CPS loss is finite
cps_a_ok = df["cps_loss_A"].apply(lambda x: x == x and abs(x) < 1e6).all()
checks.append(("cps_loss_A finite", cps_a_ok))
cps_b_ok = df["cps_loss_B"].apply(lambda x: x == x and abs(x) < 1e6).all()
checks.append(("cps_loss_B finite", cps_b_ok))

# Check 2: pseudo_agree does not saturate to 1.0
last_agree = df["pseudo_agree"].iloc[-1]
checks.append((f"pseudo_agree_fg not saturated (check manually)", True))  # global agree is always high in imbalanced segmentation

# Check 3: val_iou_A is reasonable (> 0.3 after 30 epochs)
last_val = df["val_iou_global"].iloc[-1]
checks.append((f"val_iou_A reasonable (last={last_val:.4f})", last_val > 0.3))

# Check 4: val_iou_B is reasonable (> 0.1)
last_val_B = df["val_iou_B"].iloc[-1]
checks.append((f"val_iou_B reasonable (last={last_val_B:.4f})", last_val_B > 0.1))

# Check 5: sup_loss_B > 0 from epoch 1
checks.append(("sup_loss_B > 0 all epochs", (df["sup_loss_B"] > 0).all()))

all_pass = True
for name, passed in checks:
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {name}")
    if not passed:
        all_pass = False

print(f"\n{'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED - investigate before proceeding'}")


In [ ]:
# === EVALUATE CPS SMOKE TEST (generate test_metrics + run_report with CPS fields) ===
from src.models import create_model
import torch

# Reload config from smoke test
import json as _json
_exp_dir = "/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/cps_r10_smoketest/seed_0"
with open(f"{_exp_dir}/config.json") as _f:
    cfg = _json.load(_f)
cfg["exp_dir"] = _exp_dir
cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = False

# Rebuild datasets/loaders
train_tf = get_supervised_train_augmentation(cfg)
train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
loaders = build_dataloaders(cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
                            unlabeled_ds=None, temporal_unlab_ds=None)

# Load best model and evaluate
_model = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"]).cuda()
_best = f"{_exp_dir}/best_model.pt"
_model.load_state_dict(torch.load(_best))
results = evaluate_checkpoint(cfg, _model, loaders, _best, history=None)
print(results)


In [ ]:
# === VERIFY CPS REPORTING FIELDS IN RUN_REPORT.JSON ===
import json, glob

_exp_dir = "/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/cps_r10_smoketest/seed_0"
reports = sorted(glob.glob(f"{_exp_dir}/*_run_report.json"))
if not reports:
    print("ERROR: no run_report.json found")
else:
    with open(reports[0]) as f:
        rr = json.load(f)
    
    checks = []
    
    # Check 1: ssl_method in run_identity
    ssl_m = rr.get("run_identity", {}).get("ssl_method")
    checks.append(("run_identity.ssl_method == cps", ssl_m == "cps"))
    print(f"  run_identity.ssl_method: {ssl_m}")
    
    # Check 2: cps_stats_at_best_epoch is not None
    cps_stats = rr.get("cps_stats_at_best_epoch")
    checks.append(("cps_stats_at_best_epoch is not None", cps_stats is not None))
    print(f"  cps_stats_at_best_epoch: {cps_stats}")
    
    # Check 3: pseudo_agree_fg in cps_stats
    has_fg = cps_stats is not None and "pseudo_agree_fg" in cps_stats
    checks.append(("pseudo_agree_fg in cps_stats", has_fg))
    if has_fg:
        print(f"  pseudo_agree_fg: {cps_stats['pseudo_agree_fg']}")
    
    # Check 4: diagnostic_summary.json has ssl_method
    import os
    diag_path = os.path.join(_exp_dir, "diagnostic_summary.json")
    if os.path.isfile(diag_path):
        with open(diag_path) as f:
            diag = json.load(f)
        diag_ssl = diag.get("ssl_method")
        checks.append(("diagnostic_summary.ssl_method == cps", diag_ssl == "cps"))
        print(f"  diagnostic_summary.ssl_method: {diag_ssl}")
        diag_iou_b = diag.get("best_val_iou_B")
        checks.append(("diagnostic_summary.best_val_iou_B present", diag_iou_b is not None))
        print(f"  diagnostic_summary.best_val_iou_B: {diag_iou_b}")
    else:
        print("  WARN: diagnostic_summary.json not found (smoke test used old code)")
        checks.append(("diagnostic_summary.json exists", False))
    
    # Check 5: epoch_history includes CPS columns
    eh = rr.get("epoch_history", [])
    if eh:
        last = eh[-1]
        has_cps_cols = "cps_loss_A" in last and "pseudo_agree_fg" in last
        checks.append(("epoch_history has CPS columns", has_cps_cols))
        print(f"  epoch_history last epoch cps_loss_A: {last.get('cps_loss_A')}")
        print(f"  epoch_history last epoch pseudo_agree_fg: {last.get('pseudo_agree_fg')}")
    
    print(f"
=== REPORTING VERIFICATION ===")
    all_pass = True
    for name, passed in checks:
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}")
        if not passed:
            all_pass = False
    print(f"
{'ALL REPORTING CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED'}")


## MT Non-Regression Check
Re-run MT r10 seed_0 in a fresh temp exp_dir to confirm CPS changes did not affect MT.

In [ ]:
# === MT NON-REGRESSION: mean_teacher_r10/seed_0 in fresh exp_dir ===
import gc; gc.collect()
torch.cuda.empty_cache()

cfg = get_default_config()
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

_EXP_NAME = "mean_teacher_r10_nonreg"
_SEED     = 0
cfg["exp_dir"] = f"/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/{_EXP_NAME}/seed_{_SEED}"

cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

cfg["seed"]                 = _SEED
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False
cfg["ssl_method"]           = "mean_teacher"
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 15
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0

cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 40
cfg["eval_threshold"] = 0.5

cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"

cfg["save_preds_vis"] = False
cfg["run_ruler_eval"] = True

print(summarize_config(cfg))

train_tf = get_supervised_train_augmentation(cfg)
weak_tf   = get_weak_augmentation(cfg)
strong_tf = get_strong_augmentation(cfg)

train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(cfg, weak_tf=weak_tf, strong_tf=strong_tf)
loaders = build_dataloaders(cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
                            unlabeled_ds=unlabeled_ds, temporal_unlab_ds=temporal_unlab_ds)

artifacts = run_training(cfg, loaders)

# Evaluate
from src.models import create_model
results = evaluate_checkpoint(cfg, artifacts["model"], loaders, artifacts["best_path"], artifacts["history"])
print(results)


In [ ]:
# === COMPARE MT NON-REGRESSION RESULT ===
import json, glob

# Load original MT r10 seed_0 run report
orig_dir = "/content/drive/MyDrive/UNM_vertebras_seg_v3/runs_final_v1/mean_teacher_r10/seed_0"
orig_reports = sorted(glob.glob(f"{orig_dir}/*_run_report.json"))
if orig_reports:
    with open(orig_reports[0]) as f:
        orig = json.load(f)
    orig_f1 = orig["test_metrics"]["sample_mean_f1"]
else:
    print("ERROR: original MT r10 seed_0 run_report not found")
    orig_f1 = None

# Load non-regression run report
nonreg_dir = cfg["exp_dir"]
nonreg_reports = sorted(glob.glob(f"{nonreg_dir}/*_run_report.json"))
if nonreg_reports:
    with open(nonreg_reports[0]) as f:
        nonreg = json.load(f)
    nonreg_f1 = nonreg["test_metrics"]["sample_mean_f1"]
else:
    print("ERROR: non-regression run_report not found")
    nonreg_f1 = None

if orig_f1 is not None and nonreg_f1 is not None:
    delta = abs(orig_f1 - nonreg_f1)
    print(f"Original MT r10 seed_0 F1: {orig_f1:.6f}")
    print(f"Non-reg  MT r10 seed_0 F1: {nonreg_f1:.6f}")
    print(f"Delta: {delta:.6f}")
    if delta < 0.02:
        print("PASS: MT non-regression check passed (delta < 0.02)")
    else:
        print(f"WARN: delta={delta:.6f} is large. Investigate before proceeding.")
